# LSI

### Краткое описание модели

LSI (Latent Semantic Indexing):
Основная идея: Представление документов и слов в виде векторов в пространстве с __меньшей размерностью__.
Метод: Применяется сингулярное разложение матрицы (SVD, Singular Value Decomposition) к матрице частот слов-документов. В результате получается __приближенное__ представление исходной матрицы, где сохраняется __только несколько наиболее значимых измерений (латентных тем)__.

<u>__Различия с LDA__</u>  
__Интерпретируемость__  
(LSI)Менее интерпретируемые темы.  
(LDA)Темы лучше интерпретируются благодаря вероятностной основе.  
__Производительность__  
(LSI)Быстрее для малых данных, требует меньше вычислений.	
(LDA)Может быть медленнее из-за сложных вычислений.  
__Работа с шумом__  
(LSI)Чувствительна к шуму и редким словам.  
(LDA)Более устойчива к шуму и редко встречающимся словам.  
__Объем данных__  
(LSI)Подходит для небольших и средних наборов данных.  
(LDA)Лучше работает с большими наборами данных.  

Модель полезна для:
- кластеризации документов на основе латентных тем;
- визуализации документов в низкоразмерном пространстве.;
- сравнения семантической близости документов.

### Импорт

In [12]:
import numpy as np
import pandas as pd
from pprint import pprint
import gensim
import gensim.corpora as corpora
from gensim.utils import simple_preprocess
from gensim.models import CoherenceModel

import csv
import pickle # Для сохранения словаря, корпуса и модели
import matplotlib.pyplot as plt

### 1. Загрузка данных: __data, lemma_text, dictionary, corpus__

####  - исходные данные (__data__)

In [2]:
from sklearn.datasets import fetch_20newsgroups
newsgroups_train = fetch_20newsgroups(subset='train')
data = newsgroups_train.data

In [3]:
data[1]

"From: guykuo@carson.u.washington.edu (Guy Kuo)\nSubject: SI Clock Poll - Final Call\nSummary: Final call for SI clock reports\nKeywords: SI,acceleration,clock,upgrade\nArticle-I.D.: shelley.1qvfo9INNc3s\nOrganization: University of Washington\nLines: 11\nNNTP-Posting-Host: carson.u.washington.edu\n\nA fair number of brave souls who upgraded their SI clock oscillator have\nshared their experiences for this poll. Please send a brief message detailing\nyour experiences with the procedure. Top speed attained, CPU rated speed,\nadd on cards and adapters, heat sinks, hour of usage per day, floppy disk\nfunctionality with 800 and 1.4 m floppies are especially requested.\n\nI will be summarizing in the next two days, so please add to the network\nknowledge base if you have done the clock upgrade and haven't answered this\npoll. Thanks.\n\nGuy Kuo <guykuo@u.washington.edu>\n"

#### - лемматизированный текст (__loaded_text__) 

In [13]:
with open('lemmatized_txt.csv', 'r', encoding='utf-8') as f:
    reader = csv.reader(f)
    # next(reader)  # Пропуск заголовка, если при записи мы его вставили: 
    lemma_text = [row[0] for row in reader]
    
    # with open('lemmatized_text.csv', 'w', newline='', encoding='utf-8') as f:
        # writer = csv.writer(f)
        # writer.writerow(['word'])  # Заголовок
        # for word in lemmatized_text:
            # writer.writerow([word])

In [14]:
lemma_text[4:6]

["['jcm', 'head_cfa', 'harvard', 'jonathan', 'mcdowell', 'shuttle_launch', 'question', 'organization', 'smithsonian_astrophysical', 'observatory', 'cambridge', 'usa', 'distribution', 'sci', 'line', 'article', 'owcb', 'world_std', 'com', 'tombaker', 'world_std', 'com', 'tom', 'baker', 'article', 'jlwx', 'c', 'cmu', 'etrat_ttacs', 'ttu', 'pack', 'rat', 'write', 'clear', 'caution', 'warn', 'memory', 'verify', 'unexpected', 'error', 'wonder', 'expect', 'error', 'might', 'sorry', 'really', 'dumb', 'question', 'parity_errors', 'memory', 'previously', 'know', 'condition', 'waivered', 'yes', 'error', 'already', 'knew', 'curious', 'real', 'meaning', 'quote', 'tom', 'understand', 'expected', 'error', 'basically', 'know', 'bug', 'warn', 'system', 'software', 'thing', 'check', 'right', 'value', 'yet', 'set', 'till', 'launch', 'suchlike', 'rather', 'fix', 'code', 'possibly', 'introduce', 'new', 'bug', 'tell', 'crew', 'ok', 'see', 'warn', 'liftoff', 'ignore', 'jonathan']",
 "['dfo_vttoulu', 'tko_vtt

#### - словарь (__dictionary__)

In [7]:
dictionary = corpora.Dictionary.load('dictionary.dict')

In [ ]:
pprint(dictionary.token2id)

#### - корпус (__corpus__)

In [10]:
with open('corpus.pkl', 'rb') as f:
    corpus = pickle.load(f)

In [11]:
print(type(corpus))
print(corpus[0])

<class 'list'>
[(0, 1), (1, 2), (2, 1), (3, 1), (4, 1), (5, 1), (6, 5), (7, 1), (8, 1), (9, 2), (10, 1), (11, 1), (12, 1), (13, 1), (14, 1), (15, 1), (16, 1), (17, 1), (18, 1), (19, 1), (20, 1), (21, 2), (22, 1), (23, 2), (24, 1), (25, 1), (26, 1), (27, 1), (28, 1), (29, 1), (30, 1), (31, 1), (32, 1), (33, 1), (34, 1), (35, 1), (36, 1), (37, 1), (38, 1), (39, 1), (40, 1), (41, 1), (42, 1), (43, 1), (44, 1), (45, 1), (46, 1), (47, 1), (48, 1), (49, 1), (50, 1), (51, 1)]


### 2. Построение тематической модели LSI на 6 тем (<font color='lightgreen'>lsi_model</font>)

In [15]:
lsi_model = gensim.models.lsimodel.LsiModel(
 corpus=corpus, id2word=dictionary, num_topics=6,chunksize=100
)

In [ ]:
pprint(lsi_model.print_topics())

lsi_model – преобразует текстовые данные из одного пространства (обычно мешка слов или TF-IDF) в другое пространство признаков с уменьшенной размерностью.  
lsi_model.print_topics() - выводит список латентных тем, где каждая тема состоит из множества слов с __их весами__, которые определяют, насколько они важны для этой темы.

In [17]:
doc_lsi = lsi_model[corpus]

lsi_model[corpus] – операция проекции корпуса в новое пространство признаков. Каждому документу из корпуса назначается новый вектор, где координаты представляют значения в пространстве скрытых тем (латентных семантик), выделенных моделью.

In [ ]:
for i, doc in enumerate(doc_lsi):
    print(f"Документ {i+1}:")
    for topic_id, value in doc:
        print(f"  Тема {topic_id}: {value:.4f}")

__doc_lsi__ — это список или объект-итератор, где каждый элемент соответствует одному документу из исходного corpus. Каждый документ представлен как разреженный вектор в пространстве признаков, с учетом выбранного количества латентных тем (задным параметром num_topics при создании модели LSI).  
doc_lsi может выглядеть следующим образом:  

[
    [(0, 0.8), (1, -0.5)],  # Первый документ в пространстве LSI  
    [(0, 0.3), (1, 0.6)],   # Второй документ в пространстве LSI  
    [(0, -0.7), (1, 0.2)]   # Третий документ в пространстве LSI
]  

Здесь каждая пара (i, value) означает, что значение value соответствует i-й латентной теме для данного документа.  
Причем, значения value не ограничены каким-либо диапазоном, и большие + числа соответствуют сильной связи документа с темой, и наоборот (т.е. отрицательные значения свидетельствуют об отрицательной ассоциации с темой).

### 3. Сохранение модели ((<font color='lightgreen'>'lsi_model.pkl'</font>))

In [20]:
import joblib

# Сохранение модели
joblib.dump(lsi_model, 'lsi_model.pkl')

# Загрузка модели
# loaded_model = joblib.load('lsi_model.pkl')

# Использование загруженной модели
# y_pred = loaded_model.predict(X_test)

['lsi_model.pkl']